# GMM&MMD

Detailed workflow for this experiment. For catalogue inputs and dependencies, see [data](../../../docs/data.ipynb). Enable `LG_RUN_RESEARCH_WORKFLOW=1` before running the calculation cells.

In [ ]:
import os
RUN_RESEARCH_WORKFLOW = os.environ.get("LG_RUN_RESEARCH_WORKFLOW") == "1"
if not RUN_RESEARCH_WORKFLOW:
    raise RuntimeError("Optional workflow disabled. Read notebooks/92_reproduce_saved_analyses.ipynb and set LG_RUN_RESEARCH_WORKFLOW=1.")

In [ ]:
from pathlib import Path
import os
ROOT = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / "pyproject.toml").is_file())
os.environ.setdefault("LG_PROJECT", str(ROOT / "research"))
PROJECT = Path(os.environ["LG_PROJECT"]).expanduser().resolve()
os.chdir(PROJECT)

def design_path(logical_name):
    if logical_name == "FINAL_EXPERIMENT_DESIGN_2026-10-01.md":
        return PROJECT / "out/final_stage_audit/portability_v3/original_design.txt"
    return PROJECT / logical_name

# Corrected TNG300 comparisons

Restart the kernel and run from the top. Set `LG_PROJECT` to your project directory on Roihu before running the setup cell. This notebook reads the corrected selection and existing feature caches; it does not require newly trained NPE checkpoints.

Inputs are verified against the v3 manifests. The corrected TNG sample has 1,530 pairs: 1,253 same-host and 277 different-host. Figures are written to `pickles_v3/figures/GMM&MMD`. Selection and feature caches are read only. Both simulations' radial pair velocities are **peculiar**; these comparison plots do not add a Hubble term.

Pair distributions count one row per pair. Galaxy-feature distributions explicitly distinguish unique final-pair members from pair-member entries (a galaxy may appear repeatedly). KS/energy/Wasserstein distances are descriptive; shared galaxies and hosts invalidate naive independent-row significance claims. Historical interpretation notes below require reassessment after rerunning.

Raw SOAP metadata, raw TNG catalogue checks, and large parent-catalogue diagnostics are optional flags in the setup cell and are off by default. Existing originals and outputs are preserved in the dated comparison-update backup.


In [ ]:
import os, sys
from pathlib import Path
PROJECT = Path(os.environ.get("LG_PROJECT", r"<research-project>"))
sys.path.insert(0, str(PROJECT / "analogues"))
from review_fixes import (RUN_VERSION, MODEL_DESCRIPTION, H0_BY_SIM, configure_run,
    feature_cache_dir, lg_observation, lg_to_peculiar, draw_noisy_feature,
    sample_lg_point, sample_lg_marginalized, calibration_summary, tng_volume_mpc3)
LEGACY_RESULTS_DIR = PROJECT / "pickles_v2"
FEATURE_CACHE_DIR = feature_cache_dir(PROJECT)

from comparison_utils import (load_comparison_inputs, finite_lookup,
    normalized_wasserstein, group_blocks, resample_blocks)
from splits import pair_groups, group_split
RESULTS_DIR, PLOT_DIR, tng_dir, TNG_DATA, TNG_MANIFEST = load_comparison_inputs(
    PROJECT, 'GMM&MMD')
RUN_SOAP_METADATA = False  # Optional remote metadata lookup; requires hdfstream.
RUN_TNG_CATALOGUE_CHECKS = False  # Optional raw catalogue; set TNG_BASE when enabled.
RUN_PARENT_CATALOGUE_CHECKS = False  # Optional large FLAMINGO parent-cache diagnostic.

In [ ]:
import os, sys
from pathlib import Path
PROJECT  = Path(os.environ.get('LG_PROJECT', r'<research-project>'))
TNG_BASE = os.environ.get('TNG_BASE', r'<TNG_BASE>')
sys.path.insert(0, str(PROJECT / 'analogues'))

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pickle
from pathlib import Path
from sklearn.preprocessing import StandardScaler
from sklearn.mixture import GaussianMixture
from scipy.spatial.distance import pdist

np.random.seed(42)

In [ ]:
cache_dir = PROJECT / "analogues/cache/run_001_mstar2e10"
tng_dir   = PROJECT / "analogues/tng_for_comparison_v3"
results_dir = RESULTS_DIR
results_dir.mkdir(parents=True, exist_ok=True)

with open(cache_dir / "flamingo_sample.pkl", "rb") as f:
    sample = pickle.load(f)
with open(cache_dir / "flamingo_pairs.pkl", "rb") as f:
    pairs = pickle.load(f)
with open(FEATURE_CACHE_DIR / "tng_sigma_star.pkl", "rb") as f:
    tng_sigma_star = pickle.load(f)
with open(FEATURE_CACHE_DIR / "tng_kappa_corot.pkl", "rb") as f:
    kappa_cache = pickle.load(f)
with open(FEATURE_CACHE_DIR / "metallicity_log_zscore.pkl", "rb") as f:
    met_transform = pickle.load(f)

keep_idx    = TNG_DATA["keep_idx"]
tng_pairs_i = TNG_DATA["pairs_i"]
tng_pairs_j = TNG_DATA["pairs_j"]
tng_gr_sample   = TNG_DATA["g_r"]
tng_met_sample  = TNG_DATA["metallicity"]
tng_separation  = TNG_DATA["separation"]
tng_vr          = TNG_DATA["vr"]
tng_vt          = TNG_DATA["vt"]
tng_log_m200c   = TNG_DATA["log_m200c"]
tng_log_mass_ratio = TNG_DATA["log_mass_ratio"]
tng_same_host   = TNG_DATA["have_same_host"].astype(bool)

tng_pairs_i_global = keep_idx[tng_pairs_i]
tng_pairs_j_global = keep_idx[tng_pairs_j]
tng_sigma_i = finite_lookup(tng_sigma_star, tng_pairs_i_global, "tng_sigma_star")
tng_sigma_j = finite_lookup(tng_sigma_star, tng_pairs_j_global, "tng_sigma_star")
tng_kappa_i = finite_lookup(kappa_cache, tng_pairs_i_global, "kappa_cache")
tng_kappa_j = finite_lookup(kappa_cache, tng_pairs_j_global, "kappa_cache")

In [ ]:
BLOCKS = {
    "core_dynamics":   ["separation", "vr", "vt", "log_mass_ratio"],
    "core_chemistry":  ["separation", "vr", "vt", "log_mass_ratio",
                        "met_diff"],
    "core_morphology": ["separation", "vr", "vt", "log_mass_ratio",
                        "sigma_mean", "sigma_diff", "kappa_mean", "kappa_diff"],
    "core_photometry": ["separation", "vr", "vt", "log_mass_ratio",
                        "gr_mean", "gr_diff"],
    "all_features":    ["separation", "vr", "vt", "log_mass_ratio",
                        "sigma_mean", "sigma_diff", "kappa_mean", "kappa_diff", "met_diff", "gr_mean", "gr_diff"],
}
TARGET = "log_m200c"

def pair_features_flamingo(sample, pairs, met_transform):
    pi, pj = pairs["i"], pairs["j"]
    gr_i, gr_j = sample["g_r"][pi], sample["g_r"][pj]
    sigma_i, sigma_j = sample["vel_disp"][pi], sample["vel_disp"][pj]
    kap_i, kap_j = sample["kappa_rot"][pi], sample["kappa_rot"][pj]
    met_i = met_transform["flamingo_met_dex"][pi]
    met_j = met_transform["flamingo_met_dex"][pj]
    return pd.DataFrame({
        "gr_mean": (gr_i + gr_j) / 2, "gr_diff": np.abs(gr_i - gr_j),
        "met_mean": (met_i + met_j) / 2, "met_diff": np.abs(met_i - met_j),
        "sigma_mean": (sigma_i + sigma_j) / 2, "sigma_diff": np.abs(sigma_i - sigma_j),
        "kappa_mean": (kap_i + kap_j) / 2, "kappa_diff": np.abs(kap_i - kap_j),
        "separation": pairs["separation_kpc"], "vr": pairs["vr"], "vt": pairs["vt"],
        "log_mass_ratio": pairs["log_mass_ratio"], "log_m200c": pairs["log_tot_virial_mass"],
    })

def pair_features_tng(tng_pairs_i, tng_pairs_j, tng_gr_sample, tng_met_sample,
                       tng_sigma_i, tng_sigma_j, tng_kappa_i, tng_kappa_j,
                       tng_separation, tng_vr, tng_vt, tng_log_mass_ratio,
                       tng_log_m200c, met_transform):
    gr_i, gr_j = tng_gr_sample[tng_pairs_i], tng_gr_sample[tng_pairs_j]
    met_raw_i, met_raw_j = tng_met_sample[tng_pairs_i], tng_met_sample[tng_pairs_j]
    met_i = met_transform["tng_met_dex"][tng_pairs_i]
    met_j = met_transform["tng_met_dex"][tng_pairs_j]
    return pd.DataFrame({
        "gr_mean": (gr_i + gr_j) / 2, "gr_diff": np.abs(gr_i - gr_j),
        "met_mean": (met_i + met_j) / 2, "met_diff": np.abs(met_i - met_j),
        "sigma_mean": (tng_sigma_i + tng_sigma_j) / 2, "sigma_diff": np.abs(tng_sigma_i - tng_sigma_j),
        "kappa_mean": (tng_kappa_i + tng_kappa_j) / 2, "kappa_diff": np.abs(tng_kappa_i - tng_kappa_j),
        "separation": tng_separation, "vr": tng_vr, "vt": tng_vt,
        "log_mass_ratio": tng_log_mass_ratio, "log_m200c": tng_log_m200c,
    })

flam_df = pair_features_flamingo(sample, pairs, met_transform)
tng_df  = pair_features_tng(tng_pairs_i, tng_pairs_j, tng_gr_sample, tng_met_sample,
                             tng_sigma_i, tng_sigma_j, tng_kappa_i, tng_kappa_j,
                             tng_separation, tng_vr, tng_vt, tng_log_mass_ratio,
                             tng_log_m200c, met_transform)

flam_df["same_host"] = pairs["have_same_host"].astype(bool)
tng_df["same_host"]  = tng_same_host

ALL_FEATURES = BLOCKS["all_features"]
flam_df_clean = flam_df.dropna(subset=ALL_FEATURES + [TARGET])
tng_df_clean  = tng_df.dropna(subset=ALL_FEATURES + [TARGET])
assert len(tng_df_clean) == TNG_MANIFEST["pair_count"], "Unexpected loss of corrected TNG rows."

In [ ]:
def gmm_misspecification_score(X_train, X_test, n_components=2, random_state=42):
    """Fit a GMM on X_train, score X_test's log-likelihood under it.
    Follows Huang et al. (2023), Appendix A 'Density-based approach'."""
    gmm = GaussianMixture(n_components=n_components, random_state=random_state)
    gmm.fit(X_train)
    return gmm.score_samples(X_test), gmm

In [ ]:
def rbf_kernel(X, Y, bandwidth):
    X_sq = np.sum(X**2, axis=1, keepdims=True)
    Y_sq = np.sum(Y**2, axis=1, keepdims=True)
    sq_dists = np.maximum(X_sq + Y_sq.T - 2 * X @ Y.T, 0)
    return np.exp(-sq_dists / (bandwidth**2))

def median_heuristic_bandwidth(X):
    """Median heuristic (Gretton et al. 2012), as used in the paper's Eq. 8 setup."""
    return np.sqrt(np.median(pdist(X, metric="sqeuclidean")) / 2)

def mmd_squared(X, Y, bandwidth=None, max_ref_samples=2000, random_state=42):
    """Squared MMD (V-statistic estimator, Eq. 7-8). X is subsampled to
    max_ref_samples if larger, mirroring the paper's l < m practice."""
    rng = np.random.default_rng(random_state)
    if len(X) > max_ref_samples:
        X = X[rng.choice(len(X), size=max_ref_samples, replace=False)]
    if bandwidth is None:
        bandwidth = median_heuristic_bandwidth(np.vstack([X, Y]))
    K_XX, K_YY, K_XY = rbf_kernel(X, X, bandwidth), rbf_kernel(Y, Y, bandwidth), rbf_kernel(X, Y, bandwidth)
    m, n = len(X), len(Y)
    return K_XX.sum() / m**2 + K_YY.sum() / n**2 - 2 * K_XY.sum() / (m * n)

In [ ]:
results = {}
for block_name, feats in BLOCKS.items():
    X_flam = flam_df_clean[feats].values.astype(np.float64)
    X_tng  = tng_df_clean[feats].values.astype(np.float64)

    scaler = StandardScaler().fit(X_flam)  # fit on FLAMINGO (reference dist.)
    X_flam_s, X_tng_s = scaler.transform(X_flam), scaler.transform(X_tng)

    log_liks, _ = gmm_misspecification_score(X_flam_s, X_tng_s)
    mmd2 = mmd_squared(X_flam_s, X_tng_s)

    results[block_name] = {"gmm_mean_loglik": log_liks.mean(),
                            "gmm_median_loglik": np.median(log_liks), "mmd_squared": mmd2}
    print(f"{block_name:<18} GMM mean log-lik: {log_liks.mean():>10.3f}   MMD²: {mmd2:>10.5f}")



with open(results_dir / "misspecification_scores.pkl", "wb") as f:
    pickle.dump(results, f)

In [ ]:
flam_met_dex = met_transform["flamingo_met_dex"]
tng_met_dex  = met_transform["tng_met_dex"]

flam_df_common = flam_df.copy()
tng_df_common  = tng_df.copy()

flam_df_common["met_mean"] = (flam_met_dex[pairs["i"]] + flam_met_dex[pairs["j"]]) / 2
flam_df_common["met_diff"] = np.abs(flam_met_dex[pairs["i"]] - flam_met_dex[pairs["j"]])
tng_df_common["met_mean"]  = (tng_met_dex[tng_pairs_i] + tng_met_dex[tng_pairs_j]) / 2
tng_df_common["met_diff"]  = np.abs(tng_met_dex[tng_pairs_i] - tng_met_dex[tng_pairs_j])

flam_df_common = flam_df_common.dropna(subset=ALL_FEATURES + [TARGET])
tng_df_common  = tng_df_common.dropna(subset=ALL_FEATURES + [TARGET])

print(f"FLAMINGO common-scale clean: {len(flam_df_common):,}  (original clean: {len(flam_df_clean):,})")
print(f"TNG common-scale clean:      {len(tng_df_common):,}  (original clean: {len(tng_df_clean):,})")

In [ ]:
for block_name in ["core_chemistry", "all_features"]:
    feats = BLOCKS[block_name]
    X_flam = flam_df_common[feats].values.astype(np.float64)
    X_tng  = tng_df_common[feats].values.astype(np.float64)

    scaler = StandardScaler().fit(X_flam)
    X_flam_s, X_tng_s = scaler.transform(X_flam), scaler.transform(X_tng)

    log_liks, _ = gmm_misspecification_score(X_flam_s, X_tng_s)
    mmd2 = mmd_squared(X_flam_s, X_tng_s)

    results[block_name] = {"gmm_mean_loglik": log_liks.mean(),
                            "gmm_median_loglik": np.median(log_liks), "mmd_squared": mmd2}

with open(results_dir / "misspecification_scores.pkl", "wb") as f:
    pickle.dump(results, f)

print("Final results, all blocks:")
for block_name in BLOCKS:
    r = results[block_name]
    print(f"{block_name:<18} GMM mean log-lik: {r['gmm_mean_loglik']:>10.3f}   MMD²: {r['mmd_squared']:>10.5f}")

In [ ]:
# Independent-row permutation reference for MMD^2 (exploratory).
# Shared galaxies/hosts violate row exchangeability; this is not a host-calibrated p-value.
print("MMD permutation p-values are independent-row references, not formal host-aware significance.")
N_PERM = 200

def mmd_permutation_test(X, Y, n_perm=N_PERM, max_ref_samples=2000,
                         sub_seed=42, perm_seed=7):
    """Two-sample permutation test for MMD².

    X is subsampled to max_ref_samples with the same seed mmd_squared uses, so
    the observed statistic reproduces the value from the main loop. The kernel
    bandwidth is fixed once from the pooled data and reused for every
    permutation -- otherwise the null and the observed statistic are not on the
    same scale.
    """
    rng = np.random.default_rng(sub_seed)
    if len(X) > max_ref_samples:
        X = X[rng.choice(len(X), size=max_ref_samples, replace=False)]

    bandwidth = median_heuristic_bandwidth(np.vstack([X, Y]))
    observed  = mmd_squared(X, Y, bandwidth=bandwidth)

    Z, m = np.vstack([X, Y]), len(X)
    prng = np.random.default_rng(perm_seed)
    null = np.empty(n_perm)
    for i in range(n_perm):
        idx = prng.permutation(len(Z))
        null[i] = mmd_squared(Z[idx[:m]], Z[idx[m:]], bandwidth=bandwidth)

    return observed, null, (1 + np.count_nonzero(null >= observed)) / (n_perm + 1)


perm_results = {}
print(f"{'Block':<18} {'MMD²':>10} {'null median':>13} {'null 95th':>11} {'p':>9}")
print("-" * 64)

for block_name, feats in BLOCKS.items():
    X_flam = flam_df_clean[feats].values.astype(np.float64)
    X_tng  = tng_df_clean[feats].values.astype(np.float64)
    scaler = StandardScaler().fit(X_flam)

    obs, null, p = mmd_permutation_test(scaler.transform(X_flam),
                                        scaler.transform(X_tng))

    perm_results[block_name] = {
        "mmd2": float(obs), "null_median": float(np.median(null)),
        "null_p95": float(np.percentile(null, 95)),
        "p": float(p), "n_perm": N_PERM,
        "null_assumption": "independent rows; shared-host dependence not calibrated",
    }
    print(f"{block_name:<18} {obs:>10.5f} {np.median(null):>13.5f} "
          f"{np.percentile(null, 95):>11.5f} {p:>9.4f}")

with open(results_dir / "mmd_permutation_test.pkl", "wb") as f:
    pickle.dump(perm_results, f)

In [ ]:
print(f"{'feature':<16} {'MMD²':>10} {'null 95th':>11} {'ratio':>7} {'p':>8}")
per_feature = {}
for f in BLOCKS["all_features"]:
    Xf = flam_df_clean[[f]].values.astype(np.float64)
    Yf = tng_df_clean[[f]].values.astype(np.float64)
    sc = StandardScaler().fit(Xf)
    obs, null, p = mmd_permutation_test(sc.transform(Xf), sc.transform(Yf))
    per_feature[f] = {"mmd2": float(obs), "p": float(p),
                      "null_p95": float(np.percentile(null, 95))}
    print(f"{f:<16} {obs:>10.5f} {np.percentile(null,95):>11.5f} "
          f"{obs/np.percentile(null,95):>7.1f} {p:>8.4f}")

with open(results_dir / "per_feature_mmd.pkl", "wb") as fh:
    pickle.dump(per_feature, fh)

In [ ]:
bins = np.percentile(np.concatenate([flam_df_clean["sigma_mean"],
                                     tng_df_clean["sigma_mean"]]),
                     np.linspace(0, 100, 9))
print(f"{'sigma_mean bin':>22} {'N_flam':>8} {'med θ':>8} | {'N_tng':>7} {'med θ':>8} {'Δ':>8}")
for a, b in zip(bins[:-1], bins[1:]):
    fs = flam_df_clean[(flam_df_clean.sigma_mean >= a) & (flam_df_clean.sigma_mean < b)]
    ts = tng_df_clean[(tng_df_clean.sigma_mean >= a) & (tng_df_clean.sigma_mean < b)]
    if len(ts) < 20: continue
    mf, mt = fs[TARGET].median(), ts[TARGET].median()
    print(f"{a:>10.1f}-{b:<10.1f} {len(fs):>8,} {mf:>8.3f} | {len(ts):>7,} {mt:>8.3f} {mt-mf:>+8.3f}")
print(f"\nmarginal medians — FLAMINGO {flam_df_clean[TARGET].median():.3f}, "
      f"TNG {tng_df_clean[TARGET].median():.3f}, offset {tng_df_clean[TARGET].median()-flam_df_clean[TARGET].median():+.3f}")